# Generación por imputación sobre dígitos a color

Usa el sampler `ImputationSampler` con un checkpoint **incondicional** (no requiere CFG) para reconstruir píxeles ocultados por una máscara.

**Tipos de máscara probados:**
- Banda horizontal central
- Cuadrado central
- Mitad derecha
- Píxeles aleatorios (50%)

**Demostración extra:** múltiples reconstrucciones para la misma máscara, mostrando la diversidad del espacio de soluciones plausibles.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import scipy.io
import torch
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'proyecto_AAIII_02_diffusion_models':
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from diffusion_lib import UNetScoreModelColor as ScoreNetColor
from diffusion_lib import VPProcess, CosineSchedule
from diffusion_lib.samplers.imputation import ImputationSampler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Cargar el modelo incondicional VP-Cosine

In [ ]:
process     = VPProcess(schedule=CosineSchedule())
score_model = ScoreNetColor(marginal_prob_std=process.sigma_t).to(device)

ckpt_path   = PROJECT_DIR / 'color_digits_checkpoints' / 'color_digits_VP-Cosine.pth'
score_model.load_state_dict(torch.load(ckpt_path, map_location=device))
score_model.eval()
print(f'Checkpoint cargado: {ckpt_path.name}')

## 2. Cargar imágenes reales (para tener observaciones $x_\Omega$)

Usamos algunas muestras de SVHN (mismas características que vuestro dataset de color digits).

In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

datasets.SVHN._check_integrity = lambda self: True
try:
    data = datasets.SVHN(root=str(PROJECT_DIR / 'data'), split='test',
                          download=False, transform=ToTensor())
except Exception:
    data = datasets.SVHN(root=str(PROJECT_DIR / 'data'), split='test',
                          download=True, transform=ToTensor())

loader = DataLoader(data, batch_size=4, shuffle=True)
x_real, _ = next(iter(loader))                      # [4, 3, 32, 32]
print('Forma de las imágenes:', x_real.shape, '| rango:',
      x_real.min().item(), '-', x_real.max().item())

## 3. Definir tipos de máscara

**Convención:** `mask = 1` indica píxel **conocido** (se conserva), `mask = 0` indica píxel **a inferir**.

Para imágenes a color las máscaras son por canal: `mask` tiene shape `[B, 1, H, W]` y se broadcastea sobre los 3 canales.

In [ ]:
def make_mask(shape, kind):
    B, C, H, W = shape
    m = torch.ones(B, 1, H, W)
    if kind == 'banda_horizontal':
        m[:, :, H//3 : 2*H//3, :] = 0.0
    elif kind == 'cuadrado_centro':
        m[:, :, H//4 : 3*H//4, W//4 : 3*W//4] = 0.0
    elif kind == 'mitad_derecha':
        m[:, :, :, W//2:] = 0.0
    elif kind == 'ruido_aleatorio':
        m = (torch.rand(B, 1, H, W) > 0.5).float()
    return m

MASK_TYPES = ['banda_horizontal', 'cuadrado_centro',
              'mitad_derecha', 'ruido_aleatorio']

## 4. Imputar con cada tipo de máscara

El `ImputationSampler` se inicializa con `n_resample` = número de ciclos forward/backward por paso. `n_resample=1` es estándar; valores mayores (estilo RePaint) mejoran la coherencia en el borde entre conocido y desconocido a costa de más tiempo.

In [ ]:
sampler = ImputationSampler(n_resample=1)

results = {}
for kind in MASK_TYPES:
    print(f'Imputando con máscara: {kind}...')
    m = make_mask(x_real.shape, kind).to(device)
    x_filled = sampler.inpaint(
        score_model = score_model,
        process     = process,
        x_known     = x_real.to(device),
        mask        = m,
        n_steps     = 500,
        device      = device,
        T           = 0.999,
    )
    results[kind] = (m.cpu(), x_filled.cpu())
print('Listo.')

## 5. Visualización: original / con máscara / reconstruida

In [ ]:
IDX = 0    # qué imagen del batch mostramos

fig, axes = plt.subplots(len(MASK_TYPES), 3, figsize=(7, len(MASK_TYPES)*2.4))
for i, kind in enumerate(MASK_TYPES):
    m, x_filled = results[kind]
    masked = x_real[IDX] * m[IDX]      # broadcast del canal único a 3 canales

    axes[i, 0].imshow(x_real[IDX].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[i, 1].imshow(masked.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[i, 2].imshow(x_filled[IDX].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[i, 0].set_ylabel(kind, rotation=90, fontsize=11, labelpad=10)
    for j in range(3):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

axes[0, 0].set_title('original')
axes[0, 1].set_title('máscara aplicada')
axes[0, 2].set_title('reconstrucción')
plt.tight_layout()
out_dir = PROJECT_DIR / 'figuras'
out_dir.mkdir(exist_ok=True)
fig.savefig(out_dir / 'imputacion_color_digits.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 6. Diversidad: 5 reconstrucciones para la misma máscara

Como el sampler es estocástico (Euler-Maruyama dentro de la SDE inversa), cada llamada produce una reconstrucción **distinta pero plausible**. Esto ilustra que el problema inverso es mal-puesto: hay muchas imágenes completas compatibles con la observación.

In [ ]:
N_REC = 5
KIND  = 'cuadrado_centro'

x_one = x_real[:1].to(device)
m     = make_mask(x_one.shape, KIND).to(device)

reconstructions = []
for i in range(N_REC):
    print(f'Reconstrucción {i+1}/{N_REC}...')
    x_filled = sampler.inpaint(
        score_model = score_model,
        process     = process,
        x_known     = x_one,
        mask        = m,
        n_steps     = 500,
        device      = device,
        T           = 0.999,
    )
    reconstructions.append(x_filled[0].cpu())

fig, axes = plt.subplots(1, N_REC + 2, figsize=((N_REC + 2) * 1.7, 2.2))
axes[0].imshow(x_one[0].cpu().permute(1, 2, 0).clamp(0, 1).numpy())
axes[0].set_title('original')
axes[1].imshow((x_one[0] * m[0]).cpu().permute(1, 2, 0).clamp(0, 1).numpy())
axes[1].set_title('con máscara')
for i, rec in enumerate(reconstructions):
    axes[i + 2].imshow(rec.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[i + 2].set_title(f'#{i+1}')
for ax in axes:
    ax.axis('off')
fig.suptitle(f'Diversidad de imputaciones — máscara: {KIND}', y=1.08)
plt.tight_layout()
fig.savefig(out_dir / 'imputacion_diversidad.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 7. Sanity check rápido

Verifica que los píxeles **conocidos** se conservan exactamente: $M \odot \hat x = M \odot x$, salvo error numérico despreciable.

In [ ]:
m, x_filled = results['cuadrado_centro']
x_filled = x_filled.to(device)
m        = m.to(device)
x_real_d = x_real.to(device)

resid = ((m * x_filled) - (m * x_real_d)).abs().max().item()
print(f'Máx. error en píxeles conocidos: {resid:.6f}  (debe ser pequeño)')
print(f'Píxeles conocidos: {int(m.sum().item())}  /  {m.numel()}')